In [1]:
import numpy as np

class LinearRegression:
    def __init__(self, lr=0.01, n_iterations=1000):
        self.lr = lr
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n = len(y)
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for i in range(self.n_iterations):
            y_pred = X @ self.weights + self.bias
            dL_dw = X.T @ (y_pred - y) * 2 / n
            dL_db = (y_pred - y).sum() * 2 / n
            self.weights = self.weights - self.lr * dL_dw
            self.bias = self.bias - self.lr * dL_db

    def predict(self, X):
        return X @ self.weights + self.bias

In [2]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=1000, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression(lr=0.01, n_iterations=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred):.3f}")

R^2: 0.894


In [3]:
from sklearn.linear_model import LinearRegression as SklearnLR

sk_model = SklearnLR()
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)

print(f"Sklearn R^2: {r2_score(y_test, y_pred_sk):.3f}")

Sklearn R^2: 0.894


In [4]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [12]:
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split:
            leaf_value = y.mean()
            return Node(value=leaf_value)

        feature, threshold = self._best_split(X, y)

        if feature is None:
            return Node(value=y.mean())

        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        X_left = X[left_mask]
        X_right = X[right_mask]

        y_left = y[left_mask]
        y_right = y[right_mask]

        depth += 1

        left = self._build_tree(X_left, y_left, depth)
        right = self._build_tree(X_right, y_right, depth)

        return Node(feature, threshold, left, right)

    def _best_split(self, X, y):
        n = len(y)
        best_feature, best_threshold = None, None
        best_variance_reduction = -1

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold

                y_left = y[left_mask]
                y_right = y[right_mask]

                n_left, n_right = len(y_left), len(y_right)

                if n_left == 0 or n_right == 0:
                    continue

                vr = np.var(y) - (n_left/n * np.var(y_left) + n_right/n * np.var(y_right))

                if (best_variance_reduction < vr):
                    best_variance_reduction = vr
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def predict(self, X):
        y = np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            node = self.root
            y[i] = self._traverse(X[i], node)

        return y

    def _traverse(self, x, node):
        
        if (node.value != None):
            return node.value
            
        if(x[node.feature] <= node.threshold):
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

In [8]:
X, y = make_regression(n_samples=1000, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTree(max_depth=5, min_samples_split=2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred):.3f}")

R^2: 0.602


In [9]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred):.3f}")

R^2: 0.602


In [13]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, max_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def fit(self, X, y):
        for i in range(self.n_trees):
            indices = np.random.choice(len(y), size=len(y), replace=True)
            X_sample = X[indices]
            y_sample = y[indices]
            tree = DecisionTree(self.max_depth, self.min_samples_split)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        sum_y_pred = 0
        for tree in self.trees:
            sum_y_pred += tree.predict(X)

        return sum_y_pred / self.n_trees

In [14]:
rf = RandomForest(n_trees=10, max_depth=5)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred_rf):.3f}")

R^2: 0.712
